In [ ]:
#%% 1. Seting up environment
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
pip install matplotlib

In [ ]:
pip install pillow

In [ ]:
pip install transformers

In [ ]:
pip install open3d

In [ ]:
!pip install torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --force-reinstall

In [ ]:
pip install rembg

In [ ]:
!pip install numpy==2.0.0
!pip install --upgrade numba

In [ ]:
#%% 2. Importing Libraries
import matplotlib
matplotlib.use('Agg')
from matplotlib import pyplot as pyplot
import matplotlib.pyplot as plt
from PIL import Image
import torch
from transformers import GLPNImageProcessor, GLPNForDepthEstimation
import open3d as o3d
import numpy as np
from rembg import remove
import io


In [ ]:
!pip install onnxruntime

In [ ]:
from google.colab import userdata
userdata.get('HF_Token')

In [ ]:
#%% 3. Getting Model
feature_extractor = GLPNImageProcessor.from_pretrained("vinvino02/glpn-nyu")
model = GLPNForDepthEstimation.from_pretrained("vinvino02/glpn-nyu")

In [ ]:
from google.colab import files

In [ ]:
!apt-get install libosmesa6-dev

In [ ]:
import os
os.environ['OPEN3D_USE_OSMESA'] = '1'

In [ ]:
!apt-get install xvfb
!xvfb-run -a python your_script.py

In [ ]:
#%% 4. Loading and Resizing image and also removing the background
# Upload the image file from your local machine
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

def remove_background(input_path: str) -> Image.Image:
    """Removes background from image and returns a PIL Image."""
    with open(input_path, "rb") as f:
        input_img = f.read()
    output = remove(input_img)
    return Image.open(io.BytesIO(output))

def save_image(img: Image.Image, path: str):
    img.save(path)

def image_to_array(img: Image.Image) -> np.ndarray:
    return np.array(img)

try:
    # Open the image using PIL
    image = Image.open(image_path)
    new_height = 480 if image.height > 480 else image.height
    new_height -= (new_height % 32)
    new_width = int(new_height * image.width / image.height)
    diff = new_width % 32

    new_width = new_width - diff if diff < 16 else new_width + 32 - diff
    new_size = (new_width, new_height)
    image = image.resize(new_size)

#%% 5. Preparing the image for the model
    inputs = feature_extractor(images=image, return_tensors="pt")

#%% 6. Getting the prediction from the model
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_depth = outputs.predicted_depth

    # ... (process predicted_depth) ...
except Exception as e:
    print(f"An error occurred: {e}")

#%% 7. Post processing

pad = 16
output = predicted_depth.squeeze().cpu().numpy() *1000.0
output = output[pad:-pad, pad:-pad]
image = image.crop((pad, pad, image.width - pad, image.height - pad))
output = Image.fromarray(output.astype(np.uint8))

#Visualize the prediction

fig, ax = plt.subplots(1,2)
ax[0].imshow(image)
ax[0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax[1].imshow(output, cmap = 'plasma')
ax[1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.tight_layout()
plt.pause(5)

#%% 8. Preparing the depth for open3d

width, height = image.size
output_array = np.array(output)
depth_array = (output_array*255/np.max(output_array)).astype(np.uint8)
image = np.array (image)

# Create rgbd image

depth_03d = o3d.geometry.Image(depth_array)
image_o3d = o3d.geometry.Image(image)
rgb_image = o3d.geometry.RGBDImage.create_from_color_and_depth(image_o3d, depth_03d, convert_rgb_to_intensity = False)

#%% 9. Creating a camera

camera_intrinsic = o3d.camera.PinholeCameraIntrinsic()
camera_intrinsic.set_intrinsics(width, height, 500, 500, width/2, height/2)

#%% 10. Creating o3d point cloud

pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgb_image,camera_intrinsic)

o3d.visualization.draw_geometries([pcd])

#%% 11. Post-Processing the 3d point cloud

#outlier removal
cl, ind= pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=6.0)
pcd = pcd.select_by_index(ind)

#estimate normals
pcd.estimate_normals()
pcd.orient_normals_to_align_with_direction()

o3d.visualization.draw_geometries([pcd])

#%% 12. Surface Reconstructions

mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10, n_threads=1)[0]

#Rotate the mesh
rotation = mesh.get_rotation_matrix_from_xyz((np.pi, 0,0))
center = mesh.get_center()
mesh.rotate(rotation, center=center)

#Visualize the Mesh
o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)

mesh_uniform = mesh.paint_uniform_color([1,0.706,0])
mesh_uniform.compute_vertex_normals()
o3d.visualization.draw_geometries([mesh_uniform], mesh_show_back_face=True)

o3d.visualization.draw_geometries([mesh_uniform])

#%% 13. 3d Mesh Export

#Creating in-memory buffer to store the .obj data
obj_buffer = io.BytesIO()
#Writing the mesh to the buffer
o3d.io.write_triangle_mesh(obj_buffer, mesh_uniform, write_ascii=False)
#Downloading the buffer as a .obj file
obj_buffer.seek(0)
custom_name = "my_3d_model.obj" #Change the name accordingly to what ever the desired output .obj name you would like.
files.download(obj_buffer, filename=custom_name)

